# Comparative Notebook: BiGRU vs LSTM vs ARIMA vs Transformer

Este es un **notebook comparativo**.

Su objetivo no es reentrenar modelos, sino **consolidar y comparar** los resultados ya generados por:

- `bigru_lstm_arima_univariate_recursive_multistep.ipynb`
- `transformer_univariate_recursive_multistep.ipynb`

El notebook carga los archivos Excel de métricas exportados por esos notebooks, construye tablas comparativas y genera un nuevo Excel consolidado con:

- resumen global por modelo
- resumen por variable
- resumen por iteración
- comparación por horizonte
- mejor modelo por variable
- hojas crudas con todas las filas originales

Entradas esperadas:

- `outputs_recursive_multistep/metrics_summary.xlsx`
- `outputs_recursive_multistep_transformer/metrics_summary.xlsx`

Salida principal:

- `outputs_model_comparison/model_comparison.xlsx`


## 1. Dependencias

Si el entorno no tiene las librerías necesarias, instala primero las dependencias.


In [ ]:
# Ejecuta esta celda solo si tu entorno no tiene las dependencias instaladas.
# %pip install pandas matplotlib openpyxl

## 2. Imports y configuración

Este notebook comparativo intenta leer los excels de resultados de cada familia de modelos y trabajar con cualquier subconjunto disponible.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-white")
pd.set_option("display.max_columns", 100)

NEURAL_METRICS_PATH = Path("outputs_recursive_multistep/metrics_summary.xlsx")
TRANSFORMER_METRICS_PATH = Path("outputs_recursive_multistep_transformer/metrics_summary.xlsx")

COMPARISON_OUTPUT_DIR = Path("outputs_model_comparison")
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_EXCEL_PATH = COMPARISON_OUTPUT_DIR / "model_comparison.xlsx"

EXPECTED_SHEETS = {
    NEURAL_METRICS_PATH: {
        "summary": ["BiGRU_summary", "LSTM_summary", "ARIMA_summary"],
        "horizon": ["BiGRU_horizon", "LSTM_horizon", "ARIMA_horizon"],
    },
    TRANSFORMER_METRICS_PATH: {
        "summary": ["Transformer_summary"],
        "horizon": ["Transformer_horizon"],
    },
}

print(f"Neural metrics file: {NEURAL_METRICS_PATH.resolve()}")
print(f"Transformer metrics file: {TRANSFORMER_METRICS_PATH.resolve()}")
print(f"Comparison output: {COMPARISON_EXCEL_PATH.resolve()}")

## 3. Funciones auxiliares

Se definen funciones para:

- cargar hojas disponibles desde los excels fuente
- normalizar columnas
- consolidar métricas de resumen y por horizonte
- crear tablas comparativas
- exportar el Excel comparativo


In [ ]:
def read_available_sheets(file_path: Path, expected_sheets: dict):
    status_rows = []
    loaded_summary = []
    loaded_horizon = []

    if not file_path.exists():
        for sheet_name in expected_sheets["summary"] + expected_sheets["horizon"]:
            status_rows.append({
                "file": str(file_path),
                "sheet": sheet_name,
                "status": "missing_file",
            })
        return loaded_summary, loaded_horizon, status_rows

    excel_file = pd.ExcelFile(file_path)
    available_sheets = set(excel_file.sheet_names)

    for sheet_name in expected_sheets["summary"]:
        if sheet_name in available_sheets:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
            loaded_summary.append((sheet_name, df))
            status_rows.append({
                "file": str(file_path),
                "sheet": sheet_name,
                "status": "loaded",
            })
        else:
            status_rows.append({
                "file": str(file_path),
                "sheet": sheet_name,
                "status": "missing_sheet",
            })

    for sheet_name in expected_sheets["horizon"]:
        if sheet_name in available_sheets:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
            loaded_horizon.append((sheet_name, df))
            status_rows.append({
                "file": str(file_path),
                "sheet": sheet_name,
                "status": "loaded",
            })
        else:
            status_rows.append({
                "file": str(file_path),
                "sheet": sheet_name,
                "status": "missing_sheet",
            })

    return loaded_summary, loaded_horizon, status_rows


def infer_model_name(sheet_name: str) -> str:
    if sheet_name.endswith("_summary"):
        return sheet_name.replace("_summary", "")
    if sheet_name.endswith("_horizon"):
        return sheet_name.replace("_horizon", "")
    return sheet_name


def normalize_summary_df(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    normalized = df.copy()
    normalized["model"] = model_name

    for col in ["run", "iteration", "forecast_horizon", "RMSE", "MAE", "MAPE", "training_time_seconds"]:
        if col in normalized.columns:
            normalized[col] = pd.to_numeric(normalized[col], errors="coerce")

    if "variable" in normalized.columns:
        normalized["variable"] = normalized["variable"].astype(str)

    return normalized


def normalize_horizon_df(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    normalized = df.copy()
    normalized["model"] = model_name

    for col in ["run", "iteration", "horizon", "RMSE", "MAE", "MAPE"]:
        if col in normalized.columns:
            normalized[col] = pd.to_numeric(normalized[col], errors="coerce")

    if "variable" in normalized.columns:
        normalized["variable"] = normalized["variable"].astype(str)

    return normalized


def build_overall_model_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return pd.DataFrame()

    grouped = summary_df.groupby("model", as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAPE_mean=("MAPE", "mean"),
        training_time_mean_s=("training_time_seconds", "mean"),
        records=("model", "size"),
        variables=("variable", "nunique"),
        runs=("run", "nunique"),
        iterations=("iteration", "nunique"),
    )
    return grouped.sort_values("RMSE_mean").reset_index(drop=True)


def build_by_variable_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return pd.DataFrame()

    grouped = summary_df.groupby(["variable", "model"], as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAPE_mean=("MAPE", "mean"),
        training_time_mean_s=("training_time_seconds", "mean"),
        records=("model", "size"),
    )
    return grouped.sort_values(["variable", "RMSE_mean", "model"]).reset_index(drop=True)


def build_by_iteration_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return pd.DataFrame()

    grouped = summary_df.groupby(["iteration", "model"], as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        MAE_mean=("MAE", "mean"),
        MAPE_mean=("MAPE", "mean"),
        training_time_mean_s=("training_time_seconds", "mean"),
        records=("model", "size"),
    )
    return grouped.sort_values(["iteration", "RMSE_mean", "model"]).reset_index(drop=True)


def build_best_model_per_variable(by_variable_df: pd.DataFrame) -> pd.DataFrame:
    if by_variable_df.empty:
        return pd.DataFrame()

    idx = by_variable_df.groupby("variable")["RMSE_mean"].idxmin()
    best_df = by_variable_df.loc[idx].copy()
    return best_df.sort_values("variable").reset_index(drop=True)


def build_horizon_mean_table(horizon_df: pd.DataFrame) -> pd.DataFrame:
    if horizon_df.empty:
        return pd.DataFrame()

    grouped = horizon_df.groupby(["model", "horizon"], as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        MAE_mean=("MAE", "mean"),
        MAPE_mean=("MAPE", "mean"),
        records=("model", "size"),
    )
    return grouped.sort_values(["model", "horizon"]).reset_index(drop=True)


def export_comparison_excel(output_path: Path, file_status_df: pd.DataFrame, summary_df: pd.DataFrame, horizon_df: pd.DataFrame,
                            overall_model_df: pd.DataFrame, by_variable_df: pd.DataFrame,
                            by_iteration_df: pd.DataFrame, best_model_df: pd.DataFrame,
                            horizon_mean_df: pd.DataFrame):
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        file_status_df.to_excel(writer, sheet_name="file_status", index=False)
        summary_df.to_excel(writer, sheet_name="raw_summary", index=False)
        horizon_df.to_excel(writer, sheet_name="raw_horizon", index=False)
        overall_model_df.to_excel(writer, sheet_name="overall_model", index=False)
        by_variable_df.to_excel(writer, sheet_name="by_variable", index=False)
        by_iteration_df.to_excel(writer, sheet_name="by_iteration", index=False)
        best_model_df.to_excel(writer, sheet_name="best_per_variable", index=False)
        horizon_mean_df.to_excel(writer, sheet_name="horizon_mean", index=False)


## 4. Carga de resultados fuente

Este notebook comparativo intenta cargar los excels de métricas ya generados por los notebooks de entrenamiento. Si falta alguno, el flujo continúa con los archivos disponibles.


In [ ]:
all_summary_frames = []
all_horizon_frames = []
status_rows = []

for metrics_file, sheet_groups in EXPECTED_SHEETS.items():
    loaded_summary, loaded_horizon, file_status_rows = read_available_sheets(metrics_file, sheet_groups)
    status_rows.extend(file_status_rows)

    for sheet_name, df in loaded_summary:
        model_name = infer_model_name(sheet_name)
        normalized = normalize_summary_df(model_name, df)
        if not normalized.empty:
            all_summary_frames.append(normalized)

    for sheet_name, df in loaded_horizon:
        model_name = infer_model_name(sheet_name)
        normalized = normalize_horizon_df(model_name, df)
        if not normalized.empty:
            all_horizon_frames.append(normalized)

file_status_df = pd.DataFrame(status_rows)
summary_df = pd.concat(all_summary_frames, ignore_index=True) if all_summary_frames else pd.DataFrame()
horizon_df = pd.concat(all_horizon_frames, ignore_index=True) if all_horizon_frames else pd.DataFrame()

print("Estado de archivos y hojas fuente:")
display(file_status_df)
print(f"Summary rows loaded: {len(summary_df)}")
print(f"Horizon rows loaded: {len(horizon_df)}")

## 5. Tablas comparativas

Se construyen tablas agregadas para comparar modelos a nivel global, por variable, por iteración y por horizonte.


In [ ]:
overall_model_df = build_overall_model_table(summary_df)
by_variable_df = build_by_variable_table(summary_df)
by_iteration_df = build_by_iteration_table(summary_df)
best_model_df = build_best_model_per_variable(by_variable_df)
horizon_mean_df = build_horizon_mean_table(horizon_df)

if summary_df.empty:
    print("No hay datos cargados todavía. Ejecuta primero los notebooks de entrenamiento para generar los excels fuente.")
else:
    print("Resumen global por modelo")
    display(overall_model_df)
    print("Mejor modelo por variable")
    display(best_model_df)
    print("Resumen por iteración")
    display(by_iteration_df.head(20))

## 6. Gráficas comparativas

Estas gráficas pertenecen al **notebook comparativo** y ayudan a visualizar qué modelo rinde mejor en promedio y cómo cambia el error por horizonte.


In [ ]:
if not overall_model_df.empty:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    overall_model_df.plot.bar(x="model", y="RMSE_mean", ax=axes[0], legend=False, color="#1f77b4")
    axes[0].set_title("RMSE promedio por modelo")
    axes[0].set_ylabel("RMSE")
    axes[0].tick_params(axis="x", rotation=45)

    overall_model_df.plot.bar(x="model", y="MAE_mean", ax=axes[1], legend=False, color="#ff7f0e")
    axes[1].set_title("MAE promedio por modelo")
    axes[1].set_ylabel("MAE")
    axes[1].tick_params(axis="x", rotation=45)

    overall_model_df.plot.bar(x="model", y="MAPE_mean", ax=axes[2], legend=False, color="#2ca02c")
    axes[2].set_title("MAPE promedio por modelo")
    axes[2].set_ylabel("MAPE")
    axes[2].tick_params(axis="x", rotation=45)

    overall_model_df.plot.bar(x="model", y="training_time_mean_s", ax=axes[3], legend=False, color="#d62728")
    axes[3].set_title("Tiempo promedio de entrenamiento")
    axes[3].set_ylabel("Segundos")
    axes[3].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No hay datos suficientes para graficar métricas globales.")

In [ ]:
if not horizon_mean_df.empty:
    pivot_rmse = horizon_mean_df.pivot(index="horizon", columns="model", values="RMSE_mean")
    pivot_mae = horizon_mean_df.pivot(index="horizon", columns="model", values="MAE_mean")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    pivot_rmse.plot(ax=axes[0], marker="o")
    axes[0].set_title("RMSE promedio por horizonte")
    axes[0].set_xlabel("Horizonte")
    axes[0].set_ylabel("RMSE")
    axes[0].grid(False)

    pivot_mae.plot(ax=axes[1], marker="o")
    axes[1].set_title("MAE promedio por horizonte")
    axes[1].set_xlabel("Horizonte")
    axes[1].set_ylabel("MAE")
    axes[1].grid(False)

    plt.tight_layout()
    plt.show()
else:
    print("No hay datos por horizonte para graficar.")

## 7. Exporte del Excel comparativo

El archivo consolidado de este notebook comparativo se guarda en `outputs_model_comparison/model_comparison.xlsx`.


In [ ]:
export_comparison_excel(
    output_path=COMPARISON_EXCEL_PATH,
    file_status_df=file_status_df,
    summary_df=summary_df,
    horizon_df=horizon_df,
    overall_model_df=overall_model_df,
    by_variable_df=by_variable_df,
    by_iteration_df=by_iteration_df,
    best_model_df=best_model_df,
    horizon_mean_df=horizon_mean_df,
)

print(f"Excel comparativo generado en: {COMPARISON_EXCEL_PATH.resolve()}")

## 8. Vista rápida de hojas exportadas

Esta celda permite inspeccionar rápidamente las tablas finales producidas por el notebook comparativo.


In [ ]:
display(overall_model_df)
display(best_model_df)
display(horizon_mean_df.head(20))